In [ ]:
 from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/project4_weather_agent")

folders = [
    "data/raw/pdf",
    "data/raw/markdown",
    "data/raw/web",
    "data/processed",
    "data/chunks",
    "rag",
    "tools",
    "agent",
    "evaluation",
    "notebooks"
]

for folder in folders:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Base directory:", BASE_DIR)


Project folders created successfully.
Base directory: /content/drive/MyDrive/project4_weather_agent


In [ ]:
!pip install -q langchain chromadb sentence-transformers pdfplumber beautifulsoup4 requests lxml pandas python-dotenv openai

In [ ]:
from pathlib import Path

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

PDF_DIR = RAW_DIR / "pdf"
MARKDOWN_DIR = RAW_DIR / "markdown"
WEB_DIR = RAW_DIR / "web"

PROCESSED_DIR = DATA_DIR / "processed"
CHUNKS_DIR = DATA_DIR / "chunks"

RAG_DIR = BASE_DIR / "rag"
TOOLS_DIR = BASE_DIR / "tools"
AGENT_DIR = BASE_DIR / "agent"
EVAL_DIR = BASE_DIR / "evaluation"

print("PDF_DIR:", PDF_DIR)
print("MARKDOWN_DIR:", MARKDOWN_DIR)
print("WEB_DIR:", WEB_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CHUNKS_DIR:", CHUNKS_DIR)

PDF_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/pdf
MARKDOWN_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/markdown
WEB_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/web
PROCESSED_DIR: /content/drive/MyDrive/project4_weather_agent/data/processed
CHUNKS_DIR: /content/drive/MyDrive/project4_weather_agent/data/chunks


In [ ]:
import json

config_path = BASE_DIR / "config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Config saved to:", config_path)

Config saved to: /content/drive/MyDrive/project4_weather_agent/config.json


Download diff type of files

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("/content/drive/MyDrive/project4_weather_agent")

PDF_DIR = BASE_DIR / "data/raw/pdf"
MARKDOWN_DIR = BASE_DIR / "data/raw/markdown"
WEB_DIR = BASE_DIR / "data/raw/web"

print("PDF_DIR:", PDF_DIR)
print("MARKDOWN_DIR:", MARKDOWN_DIR)
print("WEB_DIR:", WEB_DIR)

PDF_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/pdf
MARKDOWN_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/markdown
WEB_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/web


In [ ]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import os
import json
import re
import pdfplumber


In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_output_path = PDF_DIR / "noaa_climate_briefing.pdf"

response = requests.get(pdf_url, timeout=30)

if response.status_code == 200:
    with open(pdf_output_path, "wb") as f:
        f.write(response.content)
    print("PDF downloaded successfully:", pdf_output_path)
else:
    print("Failed to download PDF. Status code:", response.status_code)

PDF downloaded successfully: /content/drive/MyDrive/project4_weather_agent/data/raw/pdf/noaa_climate_briefing.pdf


In [ ]:
markdown_url = "https://raw.githubusercontent.com/open-meteo/open-meteo/main/README.md"
markdown_output_path = MARKDOWN_DIR / "open_meteo_readme.md"

response = requests.get(markdown_url, timeout=30)

if response.status_code == 200:
    with open(markdown_output_path, "w", encoding="utf-8") as f:
        f.write(response.text)
    print("Markdown downloaded successfully:", markdown_output_path)
else:
    print("Failed to download Markdown. Status code:", response.status_code)

Markdown downloaded successfully: /content/drive/MyDrive/project4_weather_agent/data/raw/markdown/open_meteo_readme.md


In [ ]:
web_pages = {
    "forecast_api.html": "https://open-meteo.com/en/docs",
    "historical_api.html": "https://open-meteo.com/en/docs/historical-weather-api"
}
for filename, url in web_pages.items():
    response = requests.get(url, timeout=30)

    if response.status_code == 200:
        output_path = WEB_DIR / filename
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"Saved: {output_path}")
    else:
        print(f"Failed to download {url} - Status code: {response.status_code}")

Saved: /content/drive/MyDrive/project4_weather_agent/data/raw/web/forecast_api.html
Saved: /content/drive/MyDrive/project4_weather_agent/data/raw/web/historical_api.html


In [ ]:
def list_files_in_folder(folder_path):
    folder_path = Path(folder_path)
    files = list(folder_path.glob("*"))
    print(f"\nFiles in {folder_path}:")
    if files:
        for file in files:
            print("-", file.name)
    else:
        print("No files found.")

list_files_in_folder(PDF_DIR)
list_files_in_folder(MARKDOWN_DIR)
list_files_in_folder(WEB_DIR)


Files in /content/drive/MyDrive/project4_weather_agent/data/raw/pdf:
- noaa_climate_briefing.pdf

Files in /content/drive/MyDrive/project4_weather_agent/data/raw/markdown:
- open_meteo_readme.md

Files in /content/drive/MyDrive/project4_weather_agent/data/raw/web:
- historical_api.html
- forecast_api.html


In [ ]:
with open(markdown_output_path, "r", encoding="utf-8") as f:
    markdown_preview = f.read()

print(markdown_preview[:500])

# 🌤 Open-Meteo Weather API

[![Test](https://github.com/open-meteo/open-meteo/actions/workflows/test.yml/badge.svg?branch=main)](https://github.com/open-meteo/open-meteo/actions/workflows/test.yml) [![GitHub license](https://img.shields.io/github/license/open-meteo/open-meteo)](https://github.com/open-meteo/open-meteo/blob/main/LICENSE) [![license: CC BY 4.0](https://img.shields.io/badge/license-CC%20BY%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/) [![Twitter](https://img.s


In [ ]:
with open(WEB_DIR / "forecast_api.html", "r", encoding="utf-8") as f:
    html_preview = f.read()
print(html_preview[:500])

<!doctype html>
<html lang="en">
	<head>
		<meta charset="utf-8" />
		<link rel="icon" href="/favicon.ico" />
		<meta name="viewport" content="width=device-width, initial-scale=1" />
		<link href="/_app/immutable/entry/start.CGthc59C.js" rel="modulepreload">
		<link href="/_app/immutable/chunks/client.BIqbh5M_.chunk.js" rel="modulepreload">
		<link href="/_app/immutable/chunks/utils.YOK87Zhf.chunk.js" rel="modulepreload">
		<link href="/_app/immutable/chunks/index-client.RiKX0zHv.chunk.js" rel="


In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/project4_weather_agent")

PDF_DIR = BASE_DIR / "data/raw/pdf"
MARKDOWN_DIR = BASE_DIR / "data/raw/markdown"
WEB_DIR = BASE_DIR / "data/raw/web"

PROCESSED_DIR = BASE_DIR / "data/processed"
CHUNKS_DIR = BASE_DIR / "data/chunks"
print("PDF_DIR:", PDF_DIR)
print("MARKDOWN_DIR:", MARKDOWN_DIR)
print("WEB_DIR:", WEB_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CHUNKS_DIR:", CHUNKS_DIR)

PDF_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/pdf
MARKDOWN_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/markdown
WEB_DIR: /content/drive/MyDrive/project4_weather_agent/data/raw/web
PROCESSED_DIR: /content/drive/MyDrive/project4_weather_agent/data/processed
CHUNKS_DIR: /content/drive/MyDrive/project4_weather_agent/data/chunks


clean text

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

extract text from pdf

In [ ]:
!pip install pymupdf

In [ ]:
import fitz  # PyMuPDF
import re

In [ ]:
def extract_text_from_pdf_pymupdf(pdf_path):
    doc = fitz.open(pdf_path)

    all_text = []

    for page in doc:
        text = page.get_text("text")  # استخراج النص
        if text:
            all_text.append(text)

    full_text = "\n".join(all_text)

    return clean_text(full_text)

In [ ]:
#TEST
pdf_files = list(PDF_DIR.glob("*.pdf"))
print("PDF files found:", [f.name for f in pdf_files])

pdf_path = pdf_files[0]

pdf_text = extract_text_from_pdf_pymupdf(pdf_path)

print("PDF preview:\n")
print(pdf_text[:1000])

PDF files found: ['noaa_climate_briefing.pdf']
PDF preview:

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms

extract text from markdown

In [ ]:
def extract_text_from_markdown(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()
    return clean_text(text)

In [ ]:
#TEST
md_files = list(MARKDOWN_DIR.glob("*.md"))
print("Markdown files found:", [f.name for f in md_files])

md_path = md_files[0]
md_text = extract_text_from_markdown(md_path)

print("Markdown preview:\n")
print(md_text[:1000])

Markdown files found: ['open_meteo_readme.md']
Markdown preview:

# 🌤 Open-Meteo Weather API [![Test](https://github.com/open-meteo/open-meteo/actions/workflows/test.yml/badge.svg?branch=main)](https://github.com/open-meteo/open-meteo/actions/workflows/test.yml) [![GitHub license](https://img.shields.io/github/license/open-meteo/open-meteo)](https://github.com/open-meteo/open-meteo/blob/main/LICENSE) [![license: CC BY 4.0](https://img.shields.io/badge/license-CC%20BY%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/) [![Twitter](https://img.shields.io/badge/follow-%40open_meteo-1DA1F2?logo=twitter&style=social)](https://twitter.com/open_meteo) [![Mastodon](https://img.shields.io/mastodon/follow/109320332765909743?domain=https%3A%2F%2Ffosstodon.org)](https://fosstodon.org/@openmeteo) [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.7970649.svg)](https://doi.org/10.5281/zenodo.7970649) Open-Meteo is an open-source weather API and offers free access for non-commercial 

In [ ]:
from IPython.display import Markdown, display

display(Markdown(md_text))

# 🌤 Open-Meteo Weather API [![Test](https://github.com/open-meteo/open-meteo/actions/workflows/test.yml/badge.svg?branch=main)](https://github.com/open-meteo/open-meteo/actions/workflows/test.yml) [![GitHub license](https://img.shields.io/github/license/open-meteo/open-meteo)](https://github.com/open-meteo/open-meteo/blob/main/LICENSE) [![license: CC BY 4.0](https://img.shields.io/badge/license-CC%20BY%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/) [![Twitter](https://img.shields.io/badge/follow-%40open_meteo-1DA1F2?logo=twitter&style=social)](https://twitter.com/open_meteo) [![Mastodon](https://img.shields.io/mastodon/follow/109320332765909743?domain=https%3A%2F%2Ffosstodon.org)](https://fosstodon.org/@openmeteo) [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.7970649.svg)](https://doi.org/10.5281/zenodo.7970649) Open-Meteo is an open-source weather API and offers free access for non-commercial use. No API key is required. You can use it immediately! Head over to https://open-meteo.com! Stay up to date with our blog at https://openmeteo.substack.com. ## Features - [Hourly weather forecast](https://open-meteo.com/en/docs) for up to 16 days - Global weather models with 11 km and regional models up to 1.5 km resolution - Weather model updates every hour for Europe and North America - 80 years [Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) - Based on the best weather models: [NOAA GFS with HRRR](https://open-meteo.com/en/docs/gfs-api), [DWD ICON](https://open-meteo.com/en/docs/dwd-api), [MeteoFrance Arome&Arpege](https://open-meteo.com/en/docs/meteofrance-api), [ECMWF IFS](https://open-meteo.com/en/docs/ecmwf-api), [JMA](https://open-meteo.com/en/docs/jma-api), [GEM HRDPS](https://open-meteo.com/en/docs/gem-api), [MET Norway](https://open-meteo.com/en/docs/metno-api) - [Marine Forecast API](https://open-meteo.com/en/docs/marine-weather-api), [Air Quality API](https://open-meteo.com/en/docs/air-quality-api), [Geocoding API](https://open-meteo.com/en/docs/geocoding-api), [Elevation API](https://open-meteo.com/en/docs/elevation-api), [Flood API](https://open-meteo.com/en/docs/flood-api) - Lightning fast APIs with response times below 10 ms - Servers located in Europe and North America with GeoDNS for best latency and high-availability - No API key required, CORS supported, no ads, no tracking, not even cookies - Free for non-commercial use with data under Attribution 4.0 International (CC BY 4.0) - Source code available under AGPLv3 ## How does Open-Meteo work? Open-Meteo utilizes open-data weather forecasts provided by national weather services. These services offer numerical weather predictions that are free to download. However, working with these models can be challenging, as it requires expertise in binary file formats, grid-systems, projections, and the fundamentals of weather predictions. Like many other weather APIs, Open-Meteo integrates high-resolution local and global weather models. Over 2 TB of data are downloaded and processed daily from multiple national weather services. The collected data is then stored in local files using a customized file format and compression technique to enhance access to time-series data such as a 14-day temperature forecast. In contrast to other weather APIs, Open-Meteo provides complete access to its source code, and all data sources are openly listed, crediting the national weather services for their work. With Docker or prebuilt Ubuntu packages, it is possible to launch your own weather API within minutes. By providing the source code, users can conduct detailed verifications of the weather data processing and even make modifications themselves. Contributions are highly encouraged and welcomed. The API is available for non-commercial use at no cost. Despite being free of charge, the forecast accuracy is top-notch. The API utilizes a vast array of local weather models with rapid updates, ensuring that the most precise forecast is generated for any location globally. ## Resources - All API documentation can be found on https://open-meteo.com. The source code for the website, documentation and API generator is available here: https://github.com/open-meteo/open-meteo-website - The free non-commerical API is hosted at [https://api.open-meteo.com](https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&hourly=temperature_2m) using to GeoDNS to servers in Europe and North America (HTTPS is optional). The API source code is in this current repository. - The geocoding API source code is available in a separate repository https://github.com/open-meteo/geocoding-api - Larger changes are announced in the [Open-Meteo Blog](https://openmeteo.substack.com) - The [Open-Meteo weather database](https://github.com/open-meteo/open-data) is redistributed as part of an AWS Open-Data Sponsorship ## Who is using Open-Meteo? Apps: - [Alpine Conditions](https://www.alpineconditions.com) Allows a user to compare multiple models at once & create ensemble forecasts for any location - [BusyRunner](https://busyrunner.com/) Allows users to plan their weekly runs based on the weather. - [Breezy Weather](https://github.com/breezy-weather/breezy-weather) A feature-rich, free and open source Material 3 Expressive Android weather app. - [Calima Canarias](https://calimacanarias.com) Real-time Saharan dust (calima) forecast and air quality monitoring for the Canary Islands and the rest of Spain. - [Cirrus](https://github.com/woheller69/omweather) Android Weather App - [Clima](https://f-droid.org/packages/co.prestosole.clima/) Beautiful, minimal, and fast weather app - [DroneWeather](https://play.google.com/store/apps/details?id=xyz.droneweather.app) Weather forecasts, satellite count, and KP index for drone pilots. - [Emojiton Weather](https://emojiton.com/weather) Get the local weather forecast for your location with fun emoji representations - [Evaporative Cooler Forecaster](https://SwampCooler.app) Swamp cooler effectiveness forecast with cost & energy savings, Android/iOS app - [FlyDecision](https://flydecision.com/) Automated weather forecast analysis and flight condition scoring for paragliding pilots, with interactive takeoff mapping. - [Home Assistant](https://www.home-assistant.io/integrations/open_meteo/) A popular open source smart home platform. - [Lively Weather](https://www.rocksdanister.com/weather) Windows native weather app powered by DirectX12 animations. - [LunaLink](https://www.lunalink.de) A site for hunters, fishermen and nature observers: It provides sun and moon values ​​(including moon brightness) as well as the weather for individual locations in Central Europe. - [Meteo-Fly](https://meteo-fly.com) Free flight-weather charts for paraglider & hang-glider pilots. - [MeteoHist](https://yotka.org/meteo-hist) A web app to create interactive temperature and precipitation graphs for places around the world - [monkeysnow](https://github.com/kcluit/monkeysnow) The most customizable resort/snow forecast website for ski and board! - [Mousam](https://amit9838.github.io/mousam/) A weather app for GNU/Linux that displays the weather at a glance - [Munetios Weather](https://weather.munetios.com) A privacy-first, non-commercial weather web app using Open-Meteo data with no tracking. - [OSS Weather](https://github.com/Akylas/oss-weather) - Multi-model/multi-provider Open Source Android/iOS Weather app - [Overmorrow](https://github.com/bmaroti9/Overmorrow) A modern material design Android weather app. - [PointWx](https://hh.guidocioni.it/pointwx/) Dash application with interactive plots (from beginner-friendly to weather-enthusiast level) easily deployable - [Precip](https://precip.ai) Hyperlocal weather history and forecast app for Android, iOS, and Web. - [QuickWeather](https://github.com/TylerWilliamson/QuickWeather) Fast, free, and open source Android app - [Rain](https://github.com/DarkMooNight/Rain) Free, open source, beautiful, minimal and fast weather app - [Raindrop](https://github.com/metalfoxdev/Raindrop) Simple and intuitive weather app for the linux terminal. - [Road Vagabond](https://roadvagabond.com) A camping destination discovery app showing zones within your drive time with weather-based filtering. - [SkyMuse](https://github.com/cakephone/skymuse) Minimal, privacy-respecting weather app. Built with web technologies. - [Slideshow](https://slideshow.digital/) Digital Signage app for Android - [solXpect](https://github.com/woheller69/solxpect) Android app which forecasts the output of your solar power plant - [The Weather](https://weather.jamesdinovo.com) A detailed, installable, progressive web application - [truthclimate](https://www.truthclimate.com) Discover how weather and climate changed all around the world. - [Typhoon](https://archisman-panigrahi.github.io/typhoon) A stylish weather app for GNU/Linux that acts as a desktop widget - [Weather Please](https://github.com/ggaidelevicius/weather-please/) Clean and minimal new tab replacement for browsers - [Weather](https://github.com/GustavLindberg99/AndroidWeather) Free, open source, simple and complete weather app for Android - [Weather.io](https://weather.roessner.tech) A simple Progressive Web App (PWA) for checking the weather. - [WeatherAI](https://play.google.com/store/apps/details?id=com.kingfu.weatherai) WeatherAI offers an intuitive user experience that makes checking the weather a breeze. - [WeatherGraph](https://weathergraph.app) Apple Watch App - [WeatherMaster](https://github.com/PranshulGG/WeatherMaster) A Weather app for android inspired by the Google Pixel weather app. - [Weatherian](https://weatherian.com/) Multi-model meteogram (multi-platform) - [weewx-DWD](https://github.com/roe-dl/weewx-DWD) Weather forecasts etc. for WeeWX - [WetBulb](https://github.com/Isma1306/wetbulb-forecast) A simple app that shows you the wetbulb temp 24h forecast and tells you if it is dangerous. - [WorldWeatherMonitor](https://world-weather-monitor.vercel.app/) An interactive world weather map that displays real-time weather conditions for cities around the globe. Repositories: - [biome](https://github.com/SqrtMinusOne/biome) Bountiful Interface to Open Meteo for Emacs - [Captain Cold](https://github.com/cburton-godaddy/captain-cold) Simple Open-Meteo -> Discord integration - [DIY Arduino esp8266 weather station](https://github.com/AlexeyMal/esp8266-weather-station) esp8266 weather station using Open-Meteo API, an embedded C++ implementation example - [Homepage](https://github.com/benphelps/homepage/) A highly customizable homepage (or startpage / application dashboard) with Docker and service API integrations. - [Spots Guru](https://www.spots.guru) Weather forecast for lazy, the best wind & wave spots around you. - [Weather-Cli](https://github.com/Rayrsn/Weather-Cli) A CLI program written in golang that allows you to get weather information from the terminal - [WeatherReport.jl](https://github.com/vnegi10/WeatherReport.jl) A simple weather app for the Julia REPL - [wthrr-the-weathercrab](https://github.com/tobealive/wthrr-the-weathercrab) Weather companion for the terminal - [weather-cli-dualprovider](https://github.com/jimishol/weather-cli-dualprovider) Minimal Bash CLI (single-file core; requires weather.en) fetching Open‑Meteo forecasts with a fallback provider; localized WMO descriptions. License: GPL‑3.0. Other: - [Menubar Weather](https://www.raycast.com/koinzhang/menubar-weather) A Raycast extension that displays live weather information in your menu bar - [MiniPavi](https://www.minipavi.fr/emulminitel/) Vintage French Minitel (a kind of BBS) weather forecast service (type "METEO" keyword on welcome Minitel screen) - [OFM-InternetWeatherModule](https://github.com/OpenKNX/OFM-InternetWeatherModule) An OpenKNX module to provide data of weather services on KNX-bus (configurable via ETS) - Contributions welcome! Do you use Open-Meteo? Please open a pull request and add your repository or app to the list! ## Client SDKs - .Net 8 / C#: https://github.com/colinnuk/open-meteo-dotnet-client-sdk - Android library for Geocoding API: https://github.com/woheller69/OmGeoDialog - Dart / Flutter: https://github.com/neursh/open-meteo-dart - Go: https://github.com/HectorMalot/omgo - Kotlin: https://github.com/open-meteo/open-meteo-api-kotlin - PHP for Geocoding API: https://gitlab.com/flibidi67/open-meteo-geocoding - PHP Laravel: https://github.com/michaelnabil230/laravel-weather - PHP Symfony 6.2: https://gitlab.com/flibidi67/open-meteo - Python: https://github.com/open-meteo/python-requests - R: https://github.com/tpisel/openmeteo - Rust: https://github.com/angelodlfrtr/open-meteo-rs - TypeScript: https://github.com/open-meteo/typescript Contributions welcome! Writing a SDK for Open-Meteo is more than welcome and a great way to help users. ## Support If you encounter bugs while using Open-Meteo APIs, please file a new issue ticket. For general ideas or Q&A please use the [Discussion](https://github.com/open-meteo/open-meteo/discussions) section on Github. Thanks! For other enquiries please contact info@open-meteo.com ## Run your own API Instructions to use Docker to run your own weather API are available in the [getting started guide](/docs/getting-started.md). ## Terms & Privacy Open-Meteo APIs are free for open-source developer and non-commercial use. We do not restrict access, but ask for fair use. If your application exceeds 10'000 requests per day, please contact us. We reserve the right to block applications and IP addresses that misuse our service. For commercial use of Open-Meteo APIs, please contact us. All data is provided as is without any warranty. We do not collect any personal data. We do not share any personal information. We do not integrate any third party analytics, ads, beacons or plugins. ## Data License API data are offered under Attribution 4.0 International (CC BY 4.0) You are free to share: copy and redistribute the material in any medium or format and adapt: remix, transform, and build upon the material. Attribution: You must give appropriate credit, provide a link to the license, and indicate if changes were made. You may do so in any reasonable manner, but not in any way that suggests the licensor endorses you or your use. You must include a link next to any location, Open-Meteo data are displayed like: <a href="https://open-meteo.com/">Weather data by Open-Meteo.com</a> ## Source Code License Open-Meteo is open-source under the GNU Affero General Public License Version 3 (AGPLv3) or any later version. You can [find the license here](LICENSE). Exceptions are third party source-code with individual licensing in each file.

# extract_text_from_html

In [ ]:
def extract_text_from_html(html_path):
    with open(html_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    return clean_text(text)

In [ ]:
html_files = list(WEB_DIR.glob("*.html"))
print("HTML files found:", [f.name for f in html_files])

html_path = html_files[0]
html_text = extract_text_from_html(html_path)

print("HTML preview:\n")
print(html_text[:1000])

HTML files found: ['historical_api.html', 'forecast_api.html']
HTML preview:

🏛️ Historical Weather API | Open-Meteo.com Open-Meteo Home Features Pricing API Docs GitHub X Toggle theme Historical Weather API Discover how weather has shaped our world from 1940 until now Historical Weather Weather Forecast Historical Forecast Previous Model Runs DWD Germany NOAA U.S. Météo-France ECMWF UK Met Office KMA Korea JMA Japan MeteoSwiss MET Norway GEM Canada BOM Australia CMA China KNMI Netherlands DMI Denmark ItaliaMeteo GeoSphere Austria Historical Weather Ensemble Models Seasonal Forecast Climate Change Marine Forecast Air Quality Satellite Radiation Geocoding Elevation Flood Now, with the addition of the 9-kilometre ECMWF IFS model, the historical weather API provides access to a staggering 90 terabytes of meteorological data! Read the blog article . Location and Time Location: Coordinates List Bounding box Latitude Longitude Not set (GMT+0) Timezone Search Start date End date You can acces

In [ ]:
documents = []

In [ ]:
for pdf_file in PDF_DIR.glob("*.pdf"):
    text = extract_text_from_pdf_pymupdf(pdf_file)
    documents.append({
        "source": pdf_file.name,
        "doc_type": "pdf",
        "content": text
    })

In [ ]:
for md_file in MARKDOWN_DIR.glob("*.md"):
    text = extract_text_from_markdown(md_file)
    documents.append({
        "source": md_file.name,
        "doc_type": "markdown",
        "content": text
    })

In [ ]:
for html_file in WEB_DIR.glob("*.html"):
    text = extract_text_from_html(html_file)
    documents.append({
        "source": html_file.name,
        "doc_type": "html",
        "content": text
    })

In [ ]:
print("Number of documents:", len(documents))
for doc in documents:
    print(f"Source: {doc['source']} | Type: {doc['doc_type']} | Length: {len(doc['content'])}")

Number of documents: 4
Source: noaa_climate_briefing.pdf | Type: pdf | Length: 39495
Source: open_meteo_readme.md | Type: markdown | Length: 14665
Source: historical_api.html | Type: html | Length: 23176
Source: forecast_api.html | Type: html | Length: 28373


In [ ]:
processed_path = PROCESSED_DIR / "processed_documents.json"

with open(processed_path, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print("Processed documents saved to:", processed_path)

Processed documents saved to: /content/drive/MyDrive/project4_weather_agent/data/processed/processed_documents.json


Chunking

In [ ]:
def chunk_text(text, chunk_size=600, overlap=120):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [ ]:
all_chunks = []
chunk_id = 0

for doc in documents:
    chunks = chunk_text(doc["content"], chunk_size=600, overlap=120)

    for chunk in chunks:
        all_chunks.append({
            "chunk_id": chunk_id,
            "source": doc["source"],
            "doc_type": doc["doc_type"],
            "content": chunk
        })
        chunk_id += 1

In [ ]:
print("Total number of chunks:", len(all_chunks))
print("\nFirst chunk preview:\n")
print(all_chunks[0]["content"][:1000])

Total number of chunks: 223

First chunk preview:

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract T


In [ ]:
chunks_path = CHUNKS_DIR / "document_chunks.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("Chunks saved to:", chunks_path)

Chunks saved to: /content/drive/MyDrive/project4_weather_agent/data/chunks/document_chunks.json


In [ ]:
print("Processed file exists:", processed_path.exists())
print("Chunks file exists:", chunks_path.exists())
print("Number of processed documents:", len(documents))
print("Number of chunks:", len(all_chunks))

Processed file exists: True
Chunks file exists: True
Number of processed documents: 4
Number of chunks: 223


embding+vectorDB

In [ ]:
!pip install -q chromadb sentence-transformers

In [ ]:
import json
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/project4_weather_agent")
CHUNKS_DIR = BASE_DIR / "data/chunks"

In [ ]:
chunks_path = CHUNKS_DIR / "document_chunks.json"

with open(chunks_path, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print("Number of loaded chunks:", len(all_chunks))
print("First chunk sample:\n")
print(all_chunks[0]["content"][:500])

Number of loaded chunks: 223
First chunk sample:

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz K


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully.


In [ ]:
sample_text = all_chunks[0]["content"]
sample_embedding = embedding_model.encode(sample_text)

print("Length of embedding vector:", len(sample_embedding))
print("First 10 values:\n", sample_embedding[:10])

Length of embedding vector: 384
First 10 values:
 [ 0.11092361  0.01662326  0.00492514  0.05424837  0.0101882   0.00703667
 -0.0345275   0.00215369 -0.00266033  0.12145294]


In [ ]:
chroma_client = chromadb.PersistentClient(path=str(BASE_DIR / "chroma_db"))
print("Chroma client created.")

Chroma client created.


In [ ]:
collection_name = "weather_project_chunks"

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(name=collection_name)
print("Collection created:", collection_name)

Collection created: weather_project_chunks


In [ ]:
documents_texts = [chunk["content"] for chunk in all_chunks]
documents_ids = [str(chunk["chunk_id"]) for chunk in all_chunks]
documents_metadata = [
    {
        "source": chunk["source"],
        "doc_type": chunk["doc_type"]
    }
    for chunk in all_chunks
]

print("Texts:", len(documents_texts))
print("IDs:", len(documents_ids))
print("Metadata:", len(documents_metadata))

Texts: 223
IDs: 223
Metadata: 223


In [ ]:
documents_embeddings = embedding_model.encode(documents_texts, show_progress_bar=True)

print("Number of embeddings:", len(documents_embeddings))
print("Embedding dimension:", len(documents_embeddings[0]))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Number of embeddings: 223
Embedding dimension: 384


In [ ]:
collection.add(
    ids=documents_ids,
    documents=documents_texts,
    embeddings=documents_embeddings.tolist(),
    metadatas=documents_metadata
)

print("All chunks added to ChromaDB successfully.")

All chunks added to ChromaDB successfully.


#test reatrival

In [ ]:
query = "What does the weather API provide?"

query_embedding = embedding_model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

In [ ]:
retrieved_docs = results["documents"][0]
retrieved_metadata = results["metadatas"][0]
retrieved_ids = results["ids"][0]

for i in range(len(retrieved_docs)):
    print(f"\nResult {i+1}")
    print("Chunk ID:", retrieved_ids[i])
    print("Source:", retrieved_metadata[i]["source"])
    print("Type:", retrieved_metadata[i]["doc_type"])
    print("Content preview:\n", retrieved_docs[i][:500])
    print("-" * 80)


Result 1
Chunk ID: 90
Source: open_meteo_readme.md
Type: markdown
Content preview:
 e code, and all data sources are openly listed, crediting the national weather services for their work. With Docker or prebuilt Ubuntu packages, it is possible to launch your own weather API within minutes. By providing the source code, users can conduct detailed verifications of the weather data processing and even make modifications themselves. Contributions are highly encouraged and welcomed. The API is available for non-commercial use at no cost. Despite being free of charge, the forecast ac
--------------------------------------------------------------------------------

Result 2
Chunk ID: 133
Source: historical_api.html
Type: html
Content preview:
 ic location and time period. To use this endpoint, you can specify a geographical coordinate, a time interval, and a list of weather variables that they are interested in. The endpoint will then return the requested data in a format that can be easily 

In [ ]:
query2 = "What is historical weather data?"

query2_embedding = embedding_model.encode(query2)

results2 = collection.query(
    query_embeddings=[query2_embedding.tolist()],
    n_results=3
)

for i in range(len(results2["documents"][0])):
    print(f"\nResult {i+1}")
    print("Source:", results2["metadatas"][0][i]["source"])
    print("Type:", results2["metadatas"][0][i]["doc_type"])
    print("Content preview:\n", results2["documents"][0][i][:500])
    print("-" * 80)


Result 1
Source: historical_api.html
Type: html
Content preview:
 ces The Historical Weather API is based on reanalysis datasets and uses a combination of weather station, aircraft, buoy, radar, and satellite observations to create a comprehensive record of past weather conditions. These datasets are able to fill in gaps by using mathematical models to estimate the values of various weather variables. As a result, reanalysis datasets are able to provide detailed historical weather information for locations that may not have had weather stations nearby, such as
--------------------------------------------------------------------------------

Result 2
Source: historical_api.html
Type: html
Content preview:
 t up-to-date version of IFS. This dataset offers the highest resolution and precision for global historical weather conditions. However, when studying climate change over decades, it is advisable to exclusively utilise ERA5 or ERA5-Land. This choice ensures data consistency and preve

use llm to genrate final answer

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
import os

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
client = OpenAI()

In [ ]:
query = "What is historical weather data?"

query_embedding = embedding_model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

retrieved_docs = results["documents"][0]
retrieved_metadata = results["metadatas"][0]

In [ ]:
context = ""

for i, doc in enumerate(retrieved_docs):
    context += f"Source {i+1}:\n{doc}\n\n"

In [ ]:
prompt = f"""
You are a helpful weather assistant.

Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}

Answer clearly and concisely.
"""

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3
)

In [ ]:
answer = response.choices[0].message.content

print("Final Answer:\n")
print(answer)

Final Answer:

Historical weather data is a comprehensive record of past weather conditions created using reanalysis datasets, which combine observations from weather stations, aircraft, buoys, radar, and satellites. These datasets can estimate weather variables for locations without nearby weather stations, providing detailed historical information.


In [ ]:
print("\nSources used:\n")

for i, meta in enumerate(retrieved_metadata):
    print(f"{i+1}. {meta['source']} ({meta['doc_type']})")


Sources used:

1. historical_api.html (html)
2. historical_api.html (html)
3. historical_api.html (html)


Api+converter+date time

In [ ]:
import requests
from datetime import datetime

In [ ]:
def get_current_weather(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    response = requests.get(url, params=params)
    data = response.json()

    return data

In [ ]:
import json

def get_current_weather(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)

        data = response.json()
        return data
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}. Response content: {response.text}")
        return {"error": f"API request failed with HTTP error: {http_err}", "raw_response": response.text}
    except json.JSONDecodeError as json_err:
        print(f"JSON decoding error: {json_err}. Raw response text: {response.text}")
        return {"error": f"Failed to decode JSON response from API: {json_err}", "raw_response": response.text}
    except requests.exceptions.RequestException as req_err:
        print(f"Request error occurred: {req_err}")
        return {"error": f"API request failed due to network or connection issue: {req_err}"}

In [ ]:
def get_weather_summary(latitude, longitude):
    data = get_current_weather(latitude, longitude)

    current = data.get("current_weather", {})

    summary = {
        "temperature": current.get("temperature"),
        "windspeed": current.get("windspeed"),
        "winddirection": current.get("winddirection"),
        "weathercode": current.get("weathercode"),
        "time": current.get("time")
    }

    return summary

In [ ]:
berlin_lat = 52.52
berlin_lon = 13.41

weather_summary = get_weather_summary(berlin_lat, berlin_lon)
weather_summary

HTTP error occurred: 502 Server Error: Bad Gateway for url: https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&current_weather=True. Response content: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>open-meteo.com | 502: Bad gateway</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-au

{'temperature': None,
 'windspeed': None,
 'winddirection': None,
 'weathercode': None,
 'time': None}

In [ ]:
def celsius_to_fahrenheit(celsius):
    return (celsius * 9/5) + 32


def fahrenheit_to_celsius(fahrenheit):
    return (fahrenheit - 32) * 5/9

In [ ]:
print("25°C to F =", celsius_to_fahrenheit(25))
print("77°F to C =", fahrenheit_to_celsius(77))

25°C to F = 77.0
77°F to C = 25.0


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def get_current_datetime():
    now = datetime.now(ZoneInfo("Asia/Riyadh"))

    return {
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "timezone": "Asia/Riyadh"
    }

In [ ]:
def rag_retrieve(query, n_results=3):
    query_embedding = embedding_model.encode(query)

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )

    retrieved_docs = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]

    output = []

    for i in range(len(retrieved_docs)):
        output.append({
            "content": retrieved_docs[i],
            "source": retrieved_metadata[i]["source"],
            "doc_type": retrieved_metadata[i]["doc_type"]
        })

    return output

In [ ]:
rag_results = rag_retrieve("What is historical weather data?")

for item in rag_results:
    print("Source:", item["source"])
    print("Type:", item["doc_type"])
    print("Preview:", item["content"][:300])
    print("-" * 80)

Source: historical_api.html
Type: html
Preview: ces The Historical Weather API is based on reanalysis datasets and uses a combination of weather station, aircraft, buoy, radar, and satellite observations to create a comprehensive record of past weather conditions. These datasets are able to fill in gaps by using mathematical models to estimate th
--------------------------------------------------------------------------------
Source: historical_api.html
Type: html
Preview: t up-to-date version of IFS. This dataset offers the highest resolution and precision for global historical weather conditions. However, when studying climate change over decades, it is advisable to exclusively utilise ERA5 or ERA5-Land. This choice ensures data consistency and prevents unintentiona
--------------------------------------------------------------------------------
Source: historical_api.html
Type: html
Preview: oil Temperature (0-100 cm) Mean Soil Temperature (0-7 cm) Mean Soil Temperature (28-100 cm) 

In [ ]:
tools = {
    "weather_api": get_weather_summary,
    "celsius_to_fahrenheit": celsius_to_fahrenheit,
    "fahrenheit_to_celsius": fahrenheit_to_celsius,
    "date_time": get_current_datetime,
    "rag_retriever": rag_retrieve
}

In [ ]:
print("Weather Tool Test:")
print(get_weather_summary(52.52, 13.41))
print()

print("Converter Tool Test:")
print("20°C -> F =", celsius_to_fahrenheit(20))
print("68°F -> C =", fahrenheit_to_celsius(68))
print()

print("Date/Time Tool Test:")
print(get_current_datetime())
print()

print("RAG Tool Test:")
rag_test = rag_retrieve("What does the weather API provide?")

for item in rag_test:
    print("Source:", item["source"])
    print("Preview:", item["content"][:200])
    print("-" * 60)

Weather Tool Test:
HTTP error occurred: 502 Server Error: Bad Gateway for url: https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&current_weather=True. Response content: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>open-meteo.com | 502: Bad gateway</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <

build agent with react 1

In [ ]:
def run_react_agent(query, max_steps=3):
    scratchpad = ""
    observations = []

    for step in range(max_steps):
        prompt = f"""
You are a ReAct-style weather assistant.

You can use these tools:
- weather_api
- rag
- converter
- date_time

Rules:
- You may call multiple tools if the question requires it.
- For comparison questions, gather all required information before answering.
- Use Thought → Action → Observation.
- Only return final_answer when you have enough information.

Question:
{query}

Previous reasoning:
{scratchpad}

Return JSON:
{{
  "thought": "...",
  "action": "weather_api or rag or converter or date_time or final_answer",
  "action_input": "..."
}}
"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        decision = json.loads(response.choices[0].message.content)

        thought = decision["thought"]
        action = decision["action"]
        action_input = decision["action_input"]

        scratchpad += f"\nThought: {thought}\n"

        if action == "final_answer":
            return {
                "final_answer": action_input,
                "scratchpad": scratchpad,
                "observations": observations
            }

        tool_result = execute_named_tool(action, action_input)

        observations.append({
            "tool": action,
            "input": action_input,
            "observation": tool_result
        })

        scratchpad += f"Action: {action}\n"
        scratchpad += f"Observation: {tool_result}\n"

    return {
        "final_answer": "I could not complete the reasoning in the allowed steps.",
        "scratchpad": scratchpad,
        "observations": observations
    }

In [ ]:
def execute_named_tool(tool_name, action_input):

    q = str(action_input).lower()

    if tool_name == "weather_api":
        city = extract_city(q)
        lat, lon = get_coordinates(city)

        if lat is None or lon is None:
            return {"error": f"Could not find coordinates for city: {city}"}

        return {
            "city": city,
            "weather_data": get_weather_summary(lat, lon)
        }

    elif tool_name == "rag":
        return crag_pipeline(q)   # نفس الشي

    elif tool_name == "converter":
        numbers = [float(word) for word in q.replace("?", "").split() if word.replace(".", "", 1).isdigit()]

        if numbers:
            value = numbers[0]

            if "celsius" in q and "fahrenheit" in q:
                return {
                    "input_celsius": value,
                    "output_fahrenheit": celsius_to_fahrenheit(value)
                }

            elif "fahrenheit" in q and "celsius" in q:
                return {
                    "input_fahrenheit": value,
                    "output_celsius": fahrenheit_to_celsius(value)
                }

        return {"error": "Conversion failed."}

    elif tool_name == "date_time":
        return get_current_datetime()

    return {"error": f"Unknown tool: {tool_name}"}

build agent with llm singel step

In [ ]:
import json
import re

def decide_tool_llm(query):
    tool_prompt = f"""
You are an intelligent weather assistant.

Available tools:
1. weather_api
   - Use for real-time weather, current weather, or forecast questions.

2. weather_and_convert
   - Use only when the user asks for BOTH weather and temperature conversion in the SAME request.

3. converter
   - Use for temperature conversion only (Celsius ↔ Fahrenheit).

4. date_time
   - Use for current date or current time questions.

5. rag
   - Use for conceptual, explanatory, or documentation-based questions.
   - Examples: definitions, explanations, historical weather data, API documentation, climate concepts.

Strict rules:
- If the query asks ONLY for conversion, choose "converter".
- If the query asks for weather AND conversion together, choose "weather_and_convert".
- If the query asks for current weather or temperature in a city, choose "weather_api".
- If the query asks for time or date, choose "date_time".
- Otherwise choose "rag".

User question:
{query}

Return ONLY valid JSON in this exact format:
{{
  "tool_name": "weather_api",
  "reason": "short explanation"
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You are a tool selection assistant. Return only valid JSON with keys tool_name and reason."
            },
            {"role": "user", "content": tool_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [ ]:
def parse_tool_decision_llm(query):
    raw_output = decide_tool_llm(query)

    allowed_tools = {"weather_api", "weather_and_convert", "converter", "date_time", "rag"}

    try:
        decision = json.loads(raw_output)
    except Exception:
        try:
            fixed_output = raw_output.replace("'", '"')
            decision = json.loads(fixed_output)
        except Exception:
            match = re.search(r'"tool_name"\s*:\s*"([^"]+)"', raw_output)
            if not match:
                match = re.search(r"'tool_name'\s*:\s*'([^']+)'", raw_output)

            tool_name = match.group(1).strip().lower() if match else None

            if tool_name in allowed_tools:
                return {
                    "tool_name": tool_name,
                    "reason": "Recovered tool_name from malformed JSON output"
                }

            q = query.lower()

            if "convert" in q and "weather" not in q:
                return {
                    "tool_name": "converter",
                    "reason": "Fallback: conversion-only query"
                }

            if "time" in q or "date" in q:
                return {
                    "tool_name": "date_time",
                    "reason": "Fallback: time/date query"
                }

            if "weather" in q and "convert" in q:
                return {
                    "tool_name": "weather_and_convert",
                    "reason": "Fallback: weather + conversion query"
                }

            if "weather" in q or "temperature" in q or "forecast" in q:
                return {
                    "tool_name": "weather_api",
                    "reason": "Fallback: weather query"
                }

            return {
                "tool_name": "rag",
                "reason": f"Fallback to rag because parsing failed. Raw output: {raw_output}"
            }

    tool_name = str(decision.get("tool_name", "")).strip().lower()
    reason = str(decision.get("reason", "")).strip()

    if tool_name not in allowed_tools:
        q = query.lower()

        if "convert" in q and "weather" not in q:
            tool_name = "converter"
            reason = "Fallback: invalid tool name from model, conversion-only query"
        elif "time" in q or "date" in q:
            tool_name = "date_time"
            reason = "Fallback: invalid tool name from model, time/date query"
        elif "weather" in q and "convert" in q:
            tool_name = "weather_and_convert"
            reason = "Fallback: invalid tool name from model, weather + conversion query"
        elif "weather" in q or "temperature" in q or "forecast" in q:
            tool_name = "weather_api"
            reason = "Fallback: invalid tool name from model, weather query"
        else:
            tool_name = "rag"
            reason = "Fallback: invalid tool name from model"

    return {
        "tool_name": tool_name,
        "reason": reason
    }

In [ ]:
def get_coordinates(city_name):
    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city_name,
        "count": 1
    }

    response = requests.get(url, params=params)
    data = response.json()

    if "results" in data:
        lat = data["results"][0]["latitude"]
        lon = data["results"][0]["longitude"]
        return lat, lon

    return None, None

In [ ]:
def extract_city(query):
    words = query.split()

    for word in words:
        if word.lower() not in ["what", "is", "the", "weather", "in", "now", "and", "convert", "it", "to", "fahrenheit"]:
            return word

    return "Berlin"

In [ ]:
def extract_city(query):
    words = query.split()

    for word in words:
        if word.lower() not in ["what", "is", "the", "weather", "in", "now"]:
            return word

    return "Berlin"

In [ ]:
def evaluate_retrieval(query, retrieved_docs):
    prompt = f"""
You are evaluating retrieval quality.

Query:
{query}

Retrieved content:
{retrieved_docs}

Is this information relevant and sufficient?

Answer ONLY:
- good
- bad
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower()

In [ ]:
def crag_pipeline(query):
    docs = rag_retrieve(query)

    evaluation = evaluate_retrieval(query, docs)

    if "good" in evaluation:
        return docs

    improved_query = query + " explanation details"

    docs_retry = rag_retrieve(improved_query)

    return docs_retry

In [ ]:
def execute_tool_llm(query, latitude=52.52, longitude=13.41):
    decision = parse_tool_decision_llm(query)
    tool_name = decision["tool_name"]
    reason = decision["reason"]

    thought = f"I analyzed the question and decided that the best tool is: {tool_name}. Reason: {reason}"

    if tool_name == "weather_api":
        city = extract_city(query)
        lat, lon = get_coordinates(city)

        action = f"Calling weather API tool for city: {city}"

        if lat is not None and lon is not None:
            observation = {
                "city": city,
                "weather_data": get_weather_summary(lat, lon)
            }
        else:
            observation = {"error": f"Could not find coordinates for city: {city}"}

    elif tool_name == "weather_and_convert":
        city = extract_city(query)
        lat, lon = get_coordinates(city)

        action = f"Calling weather API tool for city: {city}, then converting temperature to Fahrenheit"

        if lat is not None and lon is not None:
            weather = get_weather_summary(lat, lon)
            temp_c = weather.get("temperature")

            if temp_c is not None:
                temp_f = celsius_to_fahrenheit(temp_c)
                observation = {
                    "city": city,
                    "temperature_celsius": temp_c,
                    "temperature_fahrenheit": temp_f,
                    "windspeed": weather.get("windspeed"),
                    "winddirection": weather.get("winddirection"),
                    "weathercode": weather.get("weathercode"),
                    "time": weather.get("time")
                }
            else:
                observation = {"error": f"Temperature not found in weather API response for city: {city}"}
        else:
            observation = {"error": f"Could not find coordinates for city: {city}"}

    elif tool_name == "converter":
        action = "Using converter tool"

        q = query.lower()
        numbers = [float(word) for word in q.replace("?", "").split() if word.replace(".", "", 1).isdigit()]

        if numbers:
            value = numbers[0]

            if "celsius" in q and "fahrenheit" in q:
                observation = {
                    "input_celsius": value,
                    "output_fahrenheit": celsius_to_fahrenheit(value)
                }
            elif "fahrenheit" in q and "celsius" in q:
                observation = {
                    "input_fahrenheit": value,
                    "output_celsius": fahrenheit_to_celsius(value)
                }
            else:
                observation = {"error": "Could not determine conversion direction."}
        else:
            observation = {"error": "No numeric value found in the query."}

    elif tool_name == "date_time":
        action = "Using date/time tool"
        observation = get_current_datetime()

    else:
        action = "Using RAG retriever tool with corrective retrieval"
        observation = crag_pipeline(query)

    return {
        "tool_name": tool_name,
        "thought": thought,
        "action": action,
        "observation": observation
    }

In [ ]:
def build_agent_prompt(query, tool_result):
    prompt = f"""
You are a helpful weather assistant.

The user asked:
{query}

Reasoning log:
Thought: {tool_result['thought']}
Action: {tool_result['action']}
Observation: {tool_result['observation']}

Using the reasoning log above, generate a clear and concise final answer for the user.
If the answer is not in the observation, say so clearly.
If the observation contains retrieved documents, summarize them accurately.
If the observation contains weather values, explain them clearly.
If the observation contains an error, explain the problem politely.

Also mention the source when available.
"""
    return prompt

In [ ]:
chat_history = []


In [ ]:
def build_history_text(max_turns=6):
    history_text = ""
    for turn in chat_history[-max_turns:]:
        history_text += f"User: {turn['user']}\nAssistant: {turn['assistant']}\n"
    return history_text.strip()

In [ ]:
def resolve_query_with_llm_memory(query):
    history_text = build_history_text()

    prompt = f"""
You are helping a weather assistant resolve conversational references.

Your job:
- Read the conversation history.
- Read the current user query.
- If the current query depends on previous context (for example: "it", "that", "same city", "what about tomorrow"), rewrite it into a fully self-contained query.
- If the current query is already complete, return it unchanged.
- Keep the meaning exactly the same.
- Do not answer the question.
- Only rewrite the query if needed.

Conversation history:
{history_text}

Current user query:
{query}

Return ONLY valid JSON in this exact format:
{{
  "resolved_query": "fully resolved query",
  "intent_type": "new_query or follow_up",
  "reason": "short explanation"
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a query resolver for conversational memory. Return only JSON."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw_output = response.choices[0].message.content

    import json
    try:
        result = json.loads(raw_output)
        return result
    except Exception:
        return {
            "resolved_query": query,
            "intent_type": "new_query",
            "reason": f"Fallback to original query because parsing failed. Raw output: {raw_output}"
        }

In [ ]:
def build_agent_prompt_with_advanced_memory(original_query, resolved_query, intent_type, reason, tool_result):
    history_text = build_history_text()

    prompt = f"""
You are a helpful weather assistant.

Recent conversation history:
{history_text}

Original user query:
{original_query}

Resolved query:
{resolved_query}

Intent type:
{intent_type}

Resolution reason:
{reason}

Reasoning log:
Thought: {tool_result['thought']}
Action: {tool_result['action']}
Observation: {tool_result['observation']}

Instructions:
- Answer clearly and concisely.
- Use the observation only.
- Do not make up information.
- If the observation contains an error, explain it politely.
"""
    return prompt

In [ ]:
def run_agent_llm(query, latitude=52.52, longitude=13.41):
    memory_result = resolve_query_with_llm_memory(query)
    resolved_query = memory_result["resolved_query"]
    intent_type = memory_result["intent_type"]
    resolution_reason = memory_result["reason"]

    tool_result = execute_tool_llm(resolved_query, latitude=latitude, longitude=longitude)

    prompt = build_agent_prompt_with_advanced_memory(
        original_query=query,
        resolved_query=resolved_query,
        intent_type=intent_type,
        reason=resolution_reason,
        tool_result=tool_result
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful weather assistant that uses tools, conversational memory, and reasoning logs to answer user questions."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    final_answer = response.choices[0].message.content

    chat_history.append({
        "user": query,
        "assistant": final_answer
    })

    return {
        "original_query": query,
        "resolved_query": resolved_query,
        "intent_type": intent_type,
        "resolution_reason": resolution_reason,
        "tool_name": tool_result["tool_name"],
        "thought": tool_result["thought"],
        "action": tool_result["action"],
        "observation": tool_result["observation"],
        "final_answer": final_answer
    }

In [ ]:
def print_agent_log(agent_result):
    print("=" * 80)
    print("USER QUESTION:")
    print(agent_result["original_query"])
    print("-" * 80)
    print("THOUGHT:")
    print(agent_result["thought"])
    print("-" * 80)
    print("ACTION:")
    print(agent_result["action"])
    print("-" * 80)
    print("OBSERVATION:")
    print(agent_result["observation"])
    print("-" * 80)
    print("FINAL ANSWER:")
    print(agent_result["final_answer"])
    print("=" * 80)

In [ ]:
questions = [
    "What is historical weather data?",
    "What is the weather in Berlin now?",
    "Convert 20 celsius to fahrenheit",
    "What is the weather in Berlin now and convert it to fahrenheit?",
    "What time is it now?"
]

for q in questions:
    print("Question:", q)
    print(parse_tool_decision_llm(q))
    print("-" * 80)

Question: What is historical weather data?
{'tool_name': 'rag', 'reason': 'The query is asking for a conceptual explanation about historical weather data.'}
--------------------------------------------------------------------------------
Question: What is the weather in Berlin now?
{'tool_name': 'weather_api', 'reason': 'The user is asking for the current weather in Berlin.'}
--------------------------------------------------------------------------------
Question: Convert 20 celsius to fahrenheit
{'tool_name': 'converter', 'reason': 'The query asks only for temperature conversion.'}
--------------------------------------------------------------------------------
Question: What is the weather in Berlin now and convert it to fahrenheit?
{'tool_name': 'weather_and_convert', 'reason': 'The query asks for both the current weather in Berlin and a temperature conversion to Fahrenheit.'}
--------------------------------------------------------------------------------
Question: What time is it

In [ ]:
agent_result_llm_1 = run_agent_llm("What is historical weather data?")
print_agent_log(agent_result_llm_1)

USER QUESTION:
What is historical weather data?
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: rag. Reason: The query asks for historical weather data, which requires conceptual or explanatory information.
--------------------------------------------------------------------------------
ACTION:
Using RAG retriever tool with corrective retrieval
--------------------------------------------------------------------------------
OBSERVATION:
[{'content': 'ces The Historical Weather API is based on reanalysis datasets and uses a combination of weather station, aircraft, buoy, radar, and satellite observations to create a comprehensive record of past weather conditions. These datasets are able to fill in gaps by using mathematical models to estimate the values of various weather variables. As a result, reanalysis datasets are able to provide detailed historical weather information for location

In [ ]:
agent_result_llm_2 = run_agent_llm("What is the weather in Berlin now?")
print_agent_log(agent_result_llm_2)

USER QUESTION:
What is the weather in Berlin now?
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: weather_api. Reason: The query asks for the current weather in Berlin.
--------------------------------------------------------------------------------
ACTION:
Calling weather API tool for city: Berlin
--------------------------------------------------------------------------------
OBSERVATION:
{'city': 'Berlin', 'weather_data': {'temperature': 3.9, 'windspeed': 12.0, 'winddirection': 291, 'weathercode': 0, 'time': '2026-04-07T01:30'}}
--------------------------------------------------------------------------------
FINAL ANSWER:
The current weather in Berlin is as follows:
- Temperature: 3.9°C
- Windspeed: 12.0 km/h
- Wind direction: 291° (from the northwest)

If you need more information or updates, feel free to ask!


In [ ]:
agent_result_llm_3 = run_agent_llm("Convert 20 celsius to fahrenheit")
print_agent_log(agent_result_llm_3)

USER QUESTION:
Convert 20 celsius to fahrenheit
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: converter. Reason: The query asks only for temperature conversion.
--------------------------------------------------------------------------------
ACTION:
Using converter tool
--------------------------------------------------------------------------------
OBSERVATION:
{'input_celsius': 20.0, 'output_fahrenheit': 68.0}
--------------------------------------------------------------------------------
FINAL ANSWER:
20 degrees Celsius is equal to 68 degrees Fahrenheit.


In [ ]:
agent_result_llm_4 = run_agent_llm("What is the weather in Berlin now and convert it to fahrenheit?")
print_agent_log(agent_result_llm_4)

USER QUESTION:
What is the weather in Berlin now and convert it to fahrenheit?
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: weather_and_convert. Reason: The query asks for both current weather in Berlin and temperature conversion to Fahrenheit.
--------------------------------------------------------------------------------
ACTION:
Calling weather API tool for city: current, then converting temperature to Fahrenheit
--------------------------------------------------------------------------------
OBSERVATION:
{'city': 'current', 'temperature_celsius': 23.9, 'temperature_fahrenheit': 75.02, 'windspeed': 26.8, 'winddirection': 83, 'weathercode': 1, 'time': '2026-04-07T01:30'}
--------------------------------------------------------------------------------
FINAL ANSWER:
The current weather in Berlin is as follows:
- Temperature: 23.9°C (which is approximately 75.02°F)
- Windspeed: 26.8 k

In [ ]:
agent_result_llm_5 = run_agent_llm("What time is it now?")
print_agent_log(agent_result_llm_5)

USER QUESTION:
What time is it now?
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: date_time. Reason: The query asks for the current time.
--------------------------------------------------------------------------------
ACTION:
Using date/time tool
--------------------------------------------------------------------------------
OBSERVATION:
{'date': '2026-04-07', 'time': '04:36:16', 'timezone': 'Asia/Riyadh'}
--------------------------------------------------------------------------------
FINAL ANSWER:
The current time is 04:36:16 in the Asia/Riyadh timezone. If you need the time in a different timezone or have any other questions, feel free to ask!


In [ ]:
test_query = "What is the weather in Riyadh now?"

result = run_agent_llm(test_query)

print("USER QUESTION:")
print(result["original_query"])
print("\nTOOL USED:")
print(result["tool_name"])
print("\nTHOUGHT:")
print(result["thought"])
print("\nACTION:")
print(result["action"])
print("\nOBSERVATION:")
print(result["observation"])
print("\nFINAL ANSWER:")
print(result["final_answer"])

USER QUESTION:
What is the weather in Riyadh now?

TOOL USED:
weather_api

THOUGHT:
I analyzed the question and decided that the best tool is: weather_api. Reason: The query asks for the current weather in Riyadh.

ACTION:
Calling weather API tool for city: Riyadh

OBSERVATION:
{'city': 'Riyadh', 'weather_data': {'temperature': 18.0, 'windspeed': 3.6, 'winddirection': 37, 'weathercode': 1, 'time': '2026-04-07T01:30'}}

FINAL ANSWER:
The current weather in Riyadh is as follows:
- Temperature: 18.0°C
- Windspeed: 3.6 km/h
- Wind direction: 37° (from the northeast)

If you have any more questions or need further information, feel free to ask!


React 2

In [ ]:
import json
import re

def safe_json_loads(text):
    try:
        return json.loads(text)
    except Exception:
        return None


def extract_number_from_text(text):
    if text is None:
        return None

    matches = re.findall(r"-?\d+(?:\.\d+)?", str(text))
    if matches:
        return float(matches[0])
    return None


def normalize_converter_input(action_input):
    """
     dict  :
    {
        "value": 20.0,
        "from_unit": "celsius",
        "to_unit": "fahrenheit"
    }
    """

    if isinstance(action_input, dict):
        if "celsius" in action_input:
            return {
                "value": float(action_input["celsius"]),
                "from_unit": "celsius",
                "to_unit": "fahrenheit"
            }
        if "fahrenheit" in action_input:
            return {
                "value": float(action_input["fahrenheit"]),
                "from_unit": "fahrenheit",
                "to_unit": "celsius"
            }

        value = action_input.get("value")
        from_unit = action_input.get("from_unit")
        to_unit = action_input.get("to_unit")

        if value is not None and from_unit and to_unit:
            return {
                "value": float(value),
                "from_unit": str(from_unit).lower(),
                "to_unit": str(to_unit).lower()
            }

    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict):
            return normalize_converter_input(parsed)

        q = action_input.lower()

        value = extract_number_from_text(q)

        if value is None:
            return None

        if "celsius" in q and "fahrenheit" in q:
            return {
                "value": value,
                "from_unit": "celsius",
                "to_unit": "fahrenheit"
            }

        if "fahrenheit" in q and "celsius" in q:
            return {
                "value": value,
                "from_unit": "fahrenheit",
                "to_unit": "celsius"
            }

    return None


def normalize_weather_input(action_input):
    """

    """
    if isinstance(action_input, dict):
        if "city" in action_input:
            return str(action_input["city"])
        if "location" in action_input:
            return str(action_input["location"])

    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict):
            return normalize_weather_input(parsed)

        city = extract_city(action_input)
        return city

    return None


def normalize_rag_input(action_input, original_query=None):
    if isinstance(action_input, dict):
        if "query" in action_input:
            return str(action_input["query"])
        return json.dumps(action_input, ensure_ascii=False)

    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict) and "query" in parsed:
            return str(parsed["query"])
        return action_input

    return original_query if original_query else str(action_input)


def normalize_datetime_input(action_input, original_query=None):
    if isinstance(action_input, str) and action_input.strip():
        return action_input
    return original_query if original_query else "What time is it now?"


def execute_named_tool(tool_name, action_input, original_query=None):
    if tool_name == "weather_api":
        city = normalize_weather_input(action_input)

        if not city:
            city = extract_city(original_query) if original_query else None

        if not city:
            return {"error": "Could not determine the city name."}

        lat, lon = get_coordinates(city)

        if lat is None or lon is None:
            return {"error": f"Could not find coordinates for city: {city}"}

        return {
            "city": city,
            "weather_data": get_weather_summary(lat, lon)
        }

    elif tool_name == "rag":
        query_text = normalize_rag_input(action_input, original_query=original_query)
        return crag_pipeline(query_text)

    elif tool_name == "converter":
        normalized = normalize_converter_input(action_input)

        if not normalized:
            if original_query:
                normalized = normalize_converter_input(original_query)

        if not normalized:
            return {"error": "Conversion failed. Could not determine value or units."}

        value = normalized["value"]
        from_unit = normalized["from_unit"]
        to_unit = normalized["to_unit"]

        if from_unit == "celsius" and to_unit == "fahrenheit":
            return {
                "input_celsius": value,
                "output_fahrenheit": celsius_to_fahrenheit(value)
            }

        if from_unit == "fahrenheit" and to_unit == "celsius":
            return {
                "input_fahrenheit": value,
                "output_celsius": fahrenheit_to_celsius(value)
            }

        return {"error": f"Unsupported conversion: {from_unit} to {to_unit}"}

    elif tool_name == "date_time":
        _ = normalize_datetime_input(action_input, original_query=original_query)
        return get_current_datetime()

    return {"error": f"Unknown tool: {tool_name}"}


def run_react_agent(query, max_steps=3):
    scratchpad = ""
    observations = []

    for step in range(max_steps):
        prompt = f"""
You are a weather agent.

Available tools:
- weather_api
- rag
- converter
- date_time

Tool usage rules:
- Use weather_api for real-time weather questions.
- Use rag for conceptual or documentation questions.
- Use converter for temperature conversion only.
- Use date_time for current date or time.

Important:
- If using weather_api, the action_input should be the city name only.
- If using rag, the action_input should be the user query.
- If using converter, action_input can be either:
  1) a string like "Convert 20 celsius to fahrenheit"
  2) JSON like {{\"value\": 20, \"from_unit\": \"celsius\", \"to_unit\": \"fahrenheit\"}}
- If using date_time, action_input can be the same question.

Question:
{query}

Previous reasoning:
{scratchpad}

Decide the next step.

Return JSON only in this format:
{{
  "thought": "...",
  "action": "weather_api or rag or converter or date_time or final_answer",
  "action_input": "input for the tool or final answer text"
}}
"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        decision = json.loads(response.choices[0].message.content)

        thought = decision.get("thought", "")
        action = decision.get("action", "")
        action_input = decision.get("action_input", "")

        scratchpad += f"\nThought: {thought}\n"

        if action == "final_answer":
            return {
                "final_answer": action_input,
                "scratchpad": scratchpad,
                "observations": observations
            }

        tool_result = execute_named_tool(
            tool_name=action,
            action_input=action_input,
            original_query=query
        )

        observations.append({
            "tool": action,
            "input": action_input,
            "observation": tool_result
        })

        scratchpad += f"Action: {action}\n"
        scratchpad += f"Action Input: {action_input}\n"
        scratchpad += f"Observation: {tool_result}\n"

        if isinstance(tool_result, dict) and "error" in tool_result:
            continue

    return {
        "final_answer": "I could not complete the reasoning in the allowed steps.",
        "scratchpad": scratchpad,
        "observations": observations
    }

In [ ]:
result = run_react_agent("Convert 20 celsius to fahrenheit")
print(result["final_answer"])
print(result["scratchpad"])
print(result["observations"])

20 degrees Celsius is equal to 68 degrees Fahrenheit.

Thought: The user wants to convert a temperature from Celsius to Fahrenheit, so I will use the converter tool.
Action: converter
Action Input: Convert 20 celsius to fahrenheit
Observation: {'input_celsius': 20.0, 'output_fahrenheit': 68.0}

Thought: The temperature conversion from Celsius to Fahrenheit has been successfully completed, resulting in 68.0 degrees Fahrenheit.

[{'tool': 'converter', 'input': 'Convert 20 celsius to fahrenheit', 'observation': {'input_celsius': 20.0, 'output_fahrenheit': 68.0}}]


In [ ]:
result = run_react_agent("What is the weather in Riyadh now and What is historical weather data?")
print(result["final_answer"])
print(result["scratchpad"])
print(result["observations"])

HTTP error occurred: 502 Server Error: Bad Gateway for url: https://api.open-meteo.com/v1/forecast?latitude=24.68773&longitude=46.72185&current_weather=True. Response content: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx</center>
</body>
</html>

Historical weather data is based on reanalysis datasets that combine observations from weather stations, aircraft, buoys, radar, and satellites to create a comprehensive record of past weather conditions. These datasets can fill in gaps using mathematical models, providing detailed historical weather information even for locations without nearby weather stations. The data is available dating back to 1940, and for recent past weather, the Forecast API can be used with the &past_days= feature.

Thought: I need to get the current weather in Riyadh using the weather_api. For historical weather data, I will use rag to provide a conceptual answer.
Action: weather_api
Action Inp

In [ ]:
import json
import re

def safe_json_loads(text):
    try:
        return json.loads(text)
    except Exception:
        return None


def extract_number_from_text(text):
    if text is None:
        return None

    matches = re.findall(r"-?\d+(?:\.\d+)?", str(text))
    if matches:
        return float(matches[0])
    return None


def extract_city(text):
    """

    """
    if not text:
        return None

    if not isinstance(text, str):
        text = str(text)

    text = text.strip()

    if len(text.split()) <= 3 and not any(word in text.lower() for word in ["weather", "temperature", "in", "for", "what", "current"]):
        return text.title()

    patterns = [
        r"weather in ([A-Za-z\s]+)",
        r"in ([A-Za-z\s]+)",
        r"for ([A-Za-z\s]+)"
    ]

    lower_text = text.lower()

    for pattern in patterns:
        match = re.search(pattern, lower_text)
        if match:
            city = match.group(1).strip()
            city = re.sub(r"[^a-zA-Z\s]", "", city).strip()
            if city:
                return city.title()

    return None


def normalize_converter_input(action_input):
    """
       :
    {
        "value": 20.0,
        "from_unit": "celsius",
        "to_unit": "fahrenheit"
    }
    """
    if isinstance(action_input, dict):
        if "celsius" in action_input:
            return {
                "value": float(action_input["celsius"]),
                "from_unit": "celsius",
                "to_unit": "fahrenheit"
            }

        if "fahrenheit" in action_input:
            return {
                "value": float(action_input["fahrenheit"]),
                "from_unit": "fahrenheit",
                "to_unit": "celsius"
            }

        value = action_input.get("value")
        from_unit = action_input.get("from_unit")
        to_unit = action_input.get("to_unit")

        if value is not None and from_unit and to_unit:
            return {
                "value": float(value),
                "from_unit": str(from_unit).lower(),
                "to_unit": str(to_unit).lower()
            }

    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict):
            return normalize_converter_input(parsed)

        q = action_input.lower()
        value = extract_number_from_text(q)

        if value is None:
            return None

        if "celsius" in q and "fahrenheit" in q:
            return {
                "value": value,
                "from_unit": "celsius",
                "to_unit": "fahrenheit"
            }

        if "fahrenheit" in q and "celsius" in q:
            return {
                "value": value,
                "from_unit": "fahrenheit",
                "to_unit": "celsius"
            }

    return None


def normalize_weather_input(action_input):
    """

    """
    if isinstance(action_input, dict):
        if "city" in action_input:
            return str(action_input["city"]).strip()
        if "location" in action_input:
            return str(action_input["location"]).strip()

    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict):
            return normalize_weather_input(parsed)

        stripped = action_input.strip()
        if stripped and len(stripped.split()) <= 3:
            return stripped.title()

        city = extract_city(action_input)
        return city

    return None


def normalize_rag_input(action_input, original_query=None):
    if isinstance(action_input, dict):
        if "query" in action_input:
            return str(action_input["query"])
        return json.dumps(action_input, ensure_ascii=False)

    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict) and "query" in parsed:
            return str(parsed["query"])
            return action_input

    return original_query if original_query else str(action_input)


def normalize_datetime_input(action_input, original_query=None):
    if isinstance(action_input, str) and action_input.strip():
        return action_input
    return original_query if original_query else "What time is it now?"


def safe_weather_call(city):
    """
    """
    lat, lon = get_coordinates(city)

    if lat is None or lon is None:
        return {"error": f"Could not find coordinates for city: {city}"}

    try:
        result = get_weather_summary(lat, lon)

        if result is None:
            return {"error": "Weather API returned no data."}

        if isinstance(result, dict) and "error" in result:
            return result

        if isinstance(result, dict):
            values = list(result.values())
            if values and all(v is None for v in values):
                return {"error": "Weather API returned empty weather fields."}

        return {
            "city": city,
            "weather_data": result
        }

    except Exception as e:
        return {"error": f"Weather API request failed: {str(e)}"}


def execute_named_tool(tool_name, action_input, original_query=None):
    if tool_name == "weather_api":
        city = normalize_weather_input(action_input)

        if not city and original_query:
            city = extract_city(original_query)

        if not city:
            return {"error": "Could not determine the city name."}

        return safe_weather_call(city)

    elif tool_name == "rag":
        query_text = normalize_rag_input(action_input, original_query=original_query)
        return crag_pipeline(query_text)

    elif tool_name == "converter":
        normalized = normalize_converter_input(action_input)

        if not normalized and original_query:
            normalized = normalize_converter_input(original_query)

        if not normalized:
            return {"error": "Conversion failed. Could not determine value or units."}

        value = normalized["value"]
        from_unit = normalized["from_unit"]
        to_unit = normalized["to_unit"]

        if from_unit == "celsius" and to_unit == "fahrenheit":
            return {
                "input_celsius": value,
                "output_fahrenheit": celsius_to_fahrenheit(value)
            }

        if from_unit == "fahrenheit" and to_unit == "celsius":
            return {
                "input_fahrenheit": value,
                "output_celsius": fahrenheit_to_celsius(value)
            }

        return {"error": f"Unsupported conversion: {from_unit} to {to_unit}"}

    elif tool_name == "date_time":
        _ = normalize_datetime_input(action_input, original_query=original_query)
        return get_current_datetime()

    return {"error": f"Unknown tool: {tool_name}"}


def run_react_agent(query, max_steps=3):
    scratchpad = ""
    observations = []

    for step in range(max_steps):
        prompt = f"""
You are a weather agent.

Available tools:
- weather_api
- rag
- converter
- date_time

Tool usage rules:
- Use weather_api for real-time weather questions.
- Use rag for conceptual or documentation questions.
- Use converter for temperature conversion only.
- Use date_time for current date or time.

Important:
- If using weather_api, the action_input should be the city name only.
- If using rag, the action_input should be the user query.
- If using converter, action_input can be either:
  1) a string like "Convert 20 celsius to fahrenheit"
  2) JSON like {{"value": 20, "from_unit": "celsius", "to_unit": "fahrenheit"}}
- If using date_time, action_input can be the same question.
- If a tool returns an error, choose another suitable tool or provide a final answer.

Question:
{query}

Previous reasoning:
{scratchpad}

Decide the next step.

Return JSON only in this format:
{{
  "thought": "...",
  "action": "weather_api or rag or converter or date_time or final_answer",
  "action_input": "input for the tool or final answer text"
}}
"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        decision = json.loads(response.choices[0].message.content)

        thought = decision.get("thought", "")
        action = decision.get("action", "")
        action_input = decision.get("action_input", "")

        scratchpad += f"\nThought: {thought}\n"

        if action == "final_answer":
            return {
                "final_answer": action_input,
                "scratchpad": scratchpad,
                "observations": observations
            }

        tool_result = execute_named_tool(
            tool_name=action,
            action_input=action_input,
            original_query=query
        )

        observations.append({
            "tool": action,
            "input": action_input,
            "observation": tool_result
        })

        scratchpad += f"Action: {action}\n"
        scratchpad += f"Action Input: {action_input}\n"
        scratchpad += f"Observation: {tool_result}\n"

        if isinstance(tool_result, dict) and "error" in tool_result:
            continue

    return {
        "final_answer": "I could not complete the reasoning in the allowed steps.",
        "scratchpad": scratchpad,
        "observations": observations
    }

Final REact

In [ ]:
import json
import re
import requests
from datetime import datetime
from geopy.geocoders import Nominatim


# Helpers

def safe_json_loads(text):
    try:
        return json.loads(text)
    except Exception:
        return None


def extract_number_from_text(text):
    if text is None:
        return None
    matches = re.findall(r"-?\d+(?:\.\d+)?", str(text))
    if matches:
        return float(matches[0])
    return None


def extract_city(text):
    """    """
    if not text:
        return None

    patterns = [
        r"(?:weather|temperature|temp|clima|weather in|in|at|for)\s+([A-Za-z\s]+?)(?:\s*\?|$|\s+now|\s+today|\s+currently)",
        r"([A-Za-z\s]+?)\s+(?:weather|temperature|temp)",
        r"^([A-Za-z\s]+)$"
    ]

    for pattern in patterns:
        match = re.search(pattern, str(text), re.IGNORECASE)
        if match:
            city = match.group(1).strip()
            if len(city) > 1:
                return city

    return str(text).strip()

In [ ]:
#
# Coordinates

def get_coordinates(city: str):
    """ (lat, lon) ""
    try:
        geolocator = Nominatim(user_agent="weather_agent")
        location = geolocator.geocode(city, timeout=5)
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"[Geocoder Error] {e}")
    return None, None


In [ ]:
#
# Weather

def get_weather_from_open_meteo(lat: float, lon: float) -> dict:
    """: open-meteo"""
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}&current_weather=true"
    )
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    data = response.json()
    cw = data["current_weather"]
    return {
        "temperature": cw["temperature"],
        "windspeed": cw["windspeed"],
        "weathercode": cw["weathercode"],
        "source": "open-meteo"
    }


def get_weather_from_wttr(city: str) -> dict:
    """  (fallback): wttr.in"""
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    data = response.json()
    current = data["current_condition"][0]
    return {
        "temperature": float(current["temp_C"]),
        "windspeed": float(current["windspeedKmph"]),
        "weathercode": None,
        "source": "wttr.in"
    }


def get_weather_summary(lat: float, lon: float, city: str = "") -> dict:
    """
    عwttr  .
    """
    try:
        return get_weather_from_open_meteo(lat, lon)
    except Exception as e:
        print(f"[open-meteo failed] {e}")

    if city:
        try:
            return get_weather_from_wttr(city)
        except Exception as e:
            print(f"[wttr.in failed] {e}")

    return {
        "temperature": None,
        "windspeed": None,
        "weathercode": None,
        "source": "unavailable"
    }

In [ ]:
# Temperature Converter

def celsius_to_fahrenheit(c: float) -> float:
    return round((c * 9 / 5) + 32, 2)


def fahrenheit_to_celsius(f: float) -> float:
    return round((f - 32) * 5 / 9, 2)


# Date / Time

def get_current_datetime() -> dict:
    now = datetime.now()
    return {
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "day": now.strftime("%A")
    }

In [ ]:
# Normalizers

def normalize_converter_input(action_input):
    if isinstance(action_input, dict):
        if "celsius" in action_input:
            return {"value": float(action_input["celsius"]), "from_unit": "celsius", "to_unit": "fahrenheit"}
        if "fahrenheit" in action_input:
            return {"value": float(action_input["fahrenheit"]), "from_unit": "fahrenheit", "to_unit": "celsius"}
        value = action_input.get("value")
        from_unit = action_input.get("from_unit")
        to_unit = action_input.get("to_unit")
        if value is not None and from_unit and to_unit:
            return {"value": float(value), "from_unit": str(from_unit).lower(), "to_unit": str(to_unit).lower()}

    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict):
            return normalize_converter_input(parsed)
        q = action_input.lower()
        value = extract_number_from_text(q)
        if value is None:
            return None
        if "celsius" in q and "fahrenheit" in q:
            return {"value": value, "from_unit": "celsius", "to_unit": "fahrenheit"}
        if "fahrenheit" in q and "celsius" in q:
            return {"value": value, "from_unit": "fahrenheit", "to_unit": "celsius"}

    return None


def normalize_weather_input(action_input):
    if isinstance(action_input, dict):
        if "city" in action_input:
            return str(action_input["city"])
        if "location" in action_input:
            return str(action_input["location"])
    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict):
            return normalize_weather_input(parsed)
        return extract_city(action_input)
    return None


def normalize_rag_input(action_input, original_query=None):
    if isinstance(action_input, dict):
        if "query" in action_input:
            return str(action_input["query"])
        return json.dumps(action_input, ensure_ascii=False)
    if isinstance(action_input, str):
        parsed = safe_json_loads(action_input)
        if isinstance(parsed, dict) and "query" in parsed:
            return str(parsed["query"])
        return action_input
    return original_query if original_query else str(action_input)


def normalize_datetime_input(action_input, original_query=None):
    if isinstance(action_input, str) and action_input.strip():
        return action_input
    return original_query if original_query else "What time is it now?"


In [ ]:
# Tool Executor

def execute_named_tool(tool_name, action_input, original_query=None):

    if tool_name == "weather_api":
        city = normalize_weather_input(action_input)
        if not city:
            city = extract_city(original_query) if original_query else None
        if not city:
            return {"error": "Could not determine the city name."}

        lat, lon = get_coordinates(city)
        if lat is None or lon is None:
            return {"error": f"Could not find coordinates for city: {city}"}

        weather = get_weather_summary(lat, lon, city=city)

        if weather.get("temperature") is None:
            return {
                "city": city,
                "weather_data": weather,
                "error": "Weather data is currently unavailable. Please try again later."
            }

        return {"city": city, "weather_data": weather}

    elif tool_name == "rag":
        query_text = normalize_rag_input(action_input, original_query=original_query)
        return crag_pipeline(query_text)

    elif tool_name == "converter":
        normalized = normalize_converter_input(action_input)
        if not normalized and original_query:
            normalized = normalize_converter_input(original_query)
        if not normalized:
            return {"error": "Conversion failed. Could not determine value or units."}

        value, from_unit, to_unit = normalized["value"], normalized["from_unit"], normalized["to_unit"]

        if from_unit == "celsius" and to_unit == "fahrenheit":
            return {"input_celsius": value, "output_fahrenheit": celsius_to_fahrenheit(value)}
        if from_unit == "fahrenheit" and to_unit == "celsius":
            return {"input_fahrenheit": value, "output_celsius": fahrenheit_to_celsius(value)}

        return {"error": f"Unsupported conversion: {from_unit} to {to_unit}"}

    elif tool_name == "date_time":
        _ = normalize_datetime_input(action_input, original_query=original_query)
        return get_current_datetime()

    return {"error": f"Unknown tool: {tool_name}"}


In [ ]:
# ReAct Agent

def run_react_agent(query, max_steps=3):
    scratchpad = ""
    observations = []

    for step in range(max_steps):
        prompt = f"""
You are a weather agent.

Available tools:
- weather_api
- rag
- converter
- date_time

Tool usage rules:
- Use weather_api for real-time weather questions.
- Use rag for conceptual or documentation questions.
- Use converter for temperature conversion only.
- Use date_time for current date or time.

Important:
- If using weather_api, the action_input should be the city name only.
- If using rag, the action_input should be the user query.
- If using converter, action_input can be either:
  1) a string like "Convert 20 celsius to fahrenheit"
  2) JSON like {{"value": 20, "from_unit": "celsius", "to_unit": "fahrenheit"}}
- If using date_time, action_input can be the same question.
- If a tool returns an error or weather_data with null temperature,
  use rag to answer conceptually instead of retrying the same tool.

Question:
{query}

Previous reasoning:
{scratchpad}

Decide the next step.

Return JSON only in this format:
{{
  "thought": "...",
  "action": "weather_api or rag or converter or date_time or final_answer",
  "action_input": "input for the tool or final answer text"
}}
"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        decision = json.loads(response.choices[0].message.content)

        thought = decision.get("thought", "")
        action = decision.get("action", "")
        action_input = decision.get("action_input", "")

        scratchpad += f"\nThought: {thought}\n"

        if action == "final_answer":
            return {
                "final_answer": action_input,
                "scratchpad": scratchpad,
                "observations": observations
            }

        tool_result = execute_named_tool(
            tool_name=action,
            action_input=action_input,
            original_query=query
        )

        observations.append({
            "tool": action,
            "input": action_input,
            "observation": tool_result
        })

        scratchpad += f"Action: {action}\n"
        scratchpad += f"Action Input: {action_input}\n"
        scratchpad += f"Observation: {tool_result}\n"

        if isinstance(tool_result, dict) and "error" in tool_result:
            continue

    return {
        "final_answer": "I could not complete the reasoning in the allowed steps.",
        "scratchpad": scratchpad,
        "observations": observations
    }


React DEmo

In [ ]:
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

output_box = widgets.Output()

query_input = widgets.Text(
    value='',
    placeholder='Ask about weather, time, conversion, or weather knowledge...',
    description='',
    layout=widgets.Layout(width='900px', height='48px')
)

ask_button = widgets.Button(
    description='Ask',
    button_style='',
    layout=widgets.Layout(width='120px', height='48px')
)

header_html = """
<div style="
    width: 1050px;
    margin: 20px auto;
    background: white;
    border-radius: 24px;
    padding: 30px 40px;
    box-shadow: 0 8px 25px rgba(0,0,0,0.08);
    font-family: Arial, sans-serif;
">
    <div style="font-size: 42px; font-weight: 700; color: #1d4ed8; margin-bottom: 10px;">
        🌤️ Weather AI Assistant
    </div>
    <div style="font-size: 18px; color: #64748b; margin-bottom: 28px;">
        ReAct-based weather assistant for weather, conversion, time, and knowledge questions
    </div>
</div>
"""

def tool_badge(tool):
    color_map = {
        "weather_api": "#2563eb",
        "converter": "#16a34a",
        "date_time": "#f59e0b",
        "rag": "#7c3aed"
    }
    label_map = {
        "weather_api": "Weather API 🌤️",
        "converter": "Converter 🔄",
        "date_time": "Date & Time ⏰",
        "rag": "RAG + CRAG 📚"
    }
    color = color_map.get(tool, "#475569")
    label = label_map.get(tool, tool)

    return f"""
    <span style="
        display:inline-block;
        background:#eff6ff;
        color:{color};
        padding:8px 14px;
        border-radius:12px;
        font-size:16px;
        font-weight:700;
        margin-right:8px;
        margin-bottom:8px;
    ">
        {label}
    </span>
    """

def get_weather_source_badge(observations):
    for step in observations:
        if step.get("tool") == "weather_api":
            obs = step.get("observation", {})
            weather_data = obs.get("weather_data", {})
            source = weather_data.get("source", "")
            if source == "wttr.in":
                return """
                <span style="
                    display:inline-block;
                    background:#fff7ed;
                    color:#c2410c;
                    padding:6px 12px;
                    border-radius:10px;
                    font-size:14px;
                    font-weight:700;
                    margin-left:8px;
                ">
                    ⚠️ Fallback: wttr.in (open-meteo unavailable)
                </span>
                """
            elif source == "open-meteo":
                return """
                <span style="
                    display:inline-block;
                    background:#f0fdf4;
                    color:#15803d;
                    padding:6px 12px;
                    border-radius:10px;
                    font-size:14px;
                    font-weight:700;
                    margin-left:8px;
                ">
                    ✅ open-meteo
                </span>
                """
            elif source == "unavailable":
                return """
                <span style="
                    display:inline-block;
                    background:#fef2f2;
                    color:#b91c1c;
                    padding:6px 12px;
                    border-radius:10px;
                    font-size:14px;
                    font-weight:700;
                    margin-left:8px;
                ">
                    ❌ All weather sources unavailable
                </span>
                """
    return ""

def render_react_result_card(query, react_result):
    final_answer = react_result.get("final_answer", "No final answer returned.")
    observations = react_result.get("observations", [])
    scratchpad = react_result.get("scratchpad", "")

    tools_used = []
    for step in observations:
        tool = step.get("tool", "")
        if tool and tool not in tools_used:
            tools_used.append(tool)

    badges_html = "".join(tool_badge(t) for t in tools_used) if tools_used else """
    <span style="
        display:inline-block;
        background:#f1f5f9;
        color:#475569;
        padding:8px 14px;
        border-radius:12px;
        font-size:16px;
        font-weight:700;
    ">
        No tool recorded
    </span>
    """

    source_badge = get_weather_source_badge(observations)

    steps_count = len(observations)

    html = f"""
    <div style="
        width: 1050px;
        margin: 20px auto;
        background: white;
        border-radius: 24px;
        padding: 30px 40px;
        box-shadow: 0 8px 25px rgba(0,0,0,0.08);
        font-family: Arial, sans-serif;
    ">
        <div style="
            background: #dbeafe;
            color: #1e3a8a;
            padding: 18px 22px;
            border-radius: 16px;
            font-size: 24px;
            font-weight: 600;
            margin-bottom: 24px;
        ">
            {query}
        </div>

        <div style="font-size: 18px; color: #0f172a; margin-bottom: 8px; font-weight: 700;">
            Final Answer
        </div>

        <div style="
            background: #f8fafc;
            border: 1px solid #e2e8f0;
            color: #1e293b;
            padding: 22px;
            border-radius: 16px;
            font-size: 22px;
            line-height: 1.8;
            margin-bottom: 22px;
        ">
            {final_answer}
        </div>

        <div style="font-size: 18px; color: #0f172a; margin-bottom: 10px; font-weight: 700;">
            Tools Used
        </div>

        <div style="margin-bottom: 22px;">
            {badges_html}
            {source_badge}
        </div>

        <div style="
            display: inline-block;
            background: #f1f5f9;
            color: #334155;
            padding: 10px 16px;
            border-radius: 12px;
            font-size: 16px;
            font-weight: 700;
        ">
            Steps: {steps_count}
        </div>
    </div>
    """
    return html

def on_ask_clicked(b):
    query = query_input.value.strip()

    with output_box:
        clear_output()

        if not query:
            display(HTML("""
            <div style="
                width: 1050px;
                margin: 20px auto;
                background: #fff7ed;
                color: #9a3412;
                border-radius: 18px;
                padding: 22px;
                font-size: 22px;
                font-family: Arial, sans-serif;
                box-shadow: 0 8px 25px rgba(0,0,0,0.06);
            ">
                Please enter a question first.
            </div>
            """))
            return

        display(HTML("""
        <div style="
            width: 1050px;
            margin: 20px auto;
            background: white;
            border-radius: 24px;
            padding: 30px 40px;
            box-shadow: 0 8px 25px rgba(0,0,0,0.08);
            font-family: Arial, sans-serif;
            font-size: 24px;
            color: #334155;
        ">
            Running the ReAct agent...
        </div>
        """
        ))

        react_result = run_react_agent(query)

        clear_output()
        display(HTML(render_react_result_card(query, react_result)))

ask_button.on_click(on_ask_clicked)

page = widgets.VBox([
    widgets.HTML(value=header_html),
    widgets.HBox(
        [query_input, ask_button],
        layout=widgets.Layout(
            width='1050px',
            margin='0 auto 20px auto',
            justify_content='space-between'
        )
    ),
    output_box
])

display(page)


Evalution for agent use llm

In [ ]:
evaluation_questions = [
    {
        "question": "What is historical weather data?",
        "expected_tool": "rag",
        "category": "rag",
        "notes": "Conceptual documentation question"
    },
    {
        "question": "What does the weather API provide?",
        "expected_tool": "rag",
        "category": "rag",
        "notes": "Documentation / API explanation"
    },
    {
        "question": "What is the weather in Riyadh now?",
        "expected_tool": "weather_api",
        "category": "weather_api",
        "notes": "Real-time weather question"
    },
    {
        "question": "What is the weather in Tokyo now?",
        "expected_tool": "weather_api",
        "category": "weather_api",
        "notes": "Global weather support"
    },
    {
        "question": "Convert 20 celsius to fahrenheit",
        "expected_tool": "converter",
        "category": "converter",
        "notes": "Direct conversion"
    },
    {
        "question": "What time is it now?",
        "expected_tool": "date_time",
        "category": "date_time",
        "notes": "Current time question"
    }
]

In [ ]:
memory_test_conversation = [
    {
        "question": "What is the weather in Riyadh now?",
        "expected_tool": "weather_api",
        "category": "memory"
    },
    {
        "question": "Convert it to fahrenheit",
        "expected_tool": "converter",
        "category": "memory"
    },
    {
        "question": "Is it hot?",
        "expected_tool": "weather_api",
        "category": "memory"
    }
]

In [ ]:
def evaluate_single_question(item):
    question = item["question"]
    expected_tool = item["expected_tool"]

    result = run_agent_llm(question)

    used_tool = result["tool_name"]
    tool_correct = used_tool == expected_tool

    return {
        "question": question,
        "category": item["category"],
        "expected_tool": expected_tool,
        "used_tool": used_tool,
        "tool_correct": tool_correct,
        "final_answer": result["final_answer"],
        "notes": item["notes"]
    }

In [ ]:
evaluation_results = []

for item in evaluation_questions:
    row = evaluate_single_question(item)
    evaluation_results.append(row)

In [ ]:
for row in evaluation_results:
    print("=" * 100)
    print("QUESTION:", row["question"])
    print("CATEGORY:", row["category"])
    print("EXPECTED TOOL:", row["expected_tool"])
    print("USED TOOL:", row["used_tool"])
    print("TOOL CORRECT:", row["tool_correct"])
    print("ANSWER:", row["final_answer"])
    print("NOTES:", row["notes"])

QUESTION: What is historical weather data?
CATEGORY: rag
EXPECTED TOOL: rag
USED TOOL: rag
TOOL CORRECT: True
ANSWER: Historical weather data is based on reanalysis datasets that utilize a combination of observations from weather stations, aircraft, buoys, radar, and satellites to create a comprehensive record of past weather conditions. These datasets can fill in gaps by using mathematical models to estimate various weather variables, providing detailed historical weather information even for locations without nearby weather stations, such as rural areas or open oceans.

For studying climate change over decades, it is advisable to use datasets like ERA5 or ERA5-Land, which ensure data consistency and prevent unintentional alterations that could arise from different weather model upgrades. You can access historical weather data dating back to 1940.
NOTES: Conceptual documentation question
QUESTION: What does the weather API provide?
CATEGORY: rag
EXPECTED TOOL: rag
USED TOOL: rag
TOOL 

In [ ]:
correct_count = sum(1 for row in evaluation_results if row["tool_correct"])
total_count = len(evaluation_results)

tool_accuracy = correct_count / total_count

print("Tool Selection Accuracy:", round(tool_accuracy * 100, 2), "%")

Tool Selection Accuracy: 100.0 %


In [ ]:
chat_history = []

In [ ]:
memory_results = []

for item in memory_test_conversation:
    result = run_agent_llm(item["question"])

    row = {
        "question": item["question"],
        "expected_tool": item["expected_tool"],
        "used_tool": result["tool_name"],
        "tool_correct": result["tool_name"] == item["expected_tool"],
        "resolved_query": result["resolved_query"],
        "intent_type": result["intent_type"],
        "final_answer": result["final_answer"]
    }

    memory_results.append(row)

In [ ]:
for row in memory_results:
    print("=" * 100)
    print("QUESTION:", row["question"])
    print("EXPECTED TOOL:", row["expected_tool"])
    print("USED TOOL:", row["used_tool"])
    print("TOOL CORRECT:", row["tool_correct"])
    print("RESOLVED QUERY:", row["resolved_query"])
    print("INTENT TYPE:", row["intent_type"])
    print("ANSWER:", row["final_answer"])

QUESTION: What is the weather in Riyadh now?
EXPECTED TOOL: weather_api
USED TOOL: weather_api
TOOL CORRECT: True
RESOLVED QUERY: What is the weather in Riyadh now?
INTENT TYPE: new_query
ANSWER: The current weather in Riyadh is 18.0°C with a wind speed of 3.6 km/h coming from the northeast.
QUESTION: Convert it to fahrenheit
EXPECTED TOOL: converter
USED TOOL: weather_and_convert
TOOL CORRECT: False
RESOLVED QUERY: Convert the current temperature in Riyadh, which is 18.0°C, to Fahrenheit.
INTENT TYPE: follow_up
ANSWER: It seems there was an error in trying to convert the temperature. The system could not find the coordinates for the city "Convert." However, I can help you with the conversion manually.

To convert 18.0°C to Fahrenheit, you can use the formula:

°F = (°C × 9/5) + 32

So, for 18.0°C:

°F = (18.0 × 9/5) + 32 = 64.4°F

Therefore, the current temperature in Riyadh is 64.4°F.
QUESTION: Is it hot?
EXPECTED TOOL: weather_api
USED TOOL: weather_api
TOOL CORRECT: True
RESOLVED Q

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

Try Dspy framework

In [ ]:
!pip install -q dspy-ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.

In [ ]:
import dspy
import json
import os

In [ ]:
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)

In [ ]:
class ToolSelector(dspy.Signature):
    """
    Select the best tool for a weather assistant query.
    """
    query = dspy.InputField()
    tool_name = dspy.OutputField(desc="One of: weather_api, weather_and_convert, converter, date_time, rag")
    reason = dspy.OutputField(desc="Short explanation for why this tool is the best choice")

In [ ]:
class WeatherResponder(dspy.Signature):
    """
    Generate a final answer for the user based on the selected tool result.
    """
    query = dspy.InputField()
    thought = dspy.InputField()
    action = dspy.InputField()
    observation = dspy.InputField()
    final_answer = dspy.OutputField(desc="A clear and concise answer for the user")

In [ ]:
tool_selector = dspy.Predict(ToolSelector)
weather_responder = dspy.Predict(WeatherResponder)

In [ ]:
def decide_tool_dspy(query):
    result = tool_selector(query=query)

    tool_name = str(result.tool_name).strip().lower()
    reason = str(result.reason).strip()

    allowed_tools = {"weather_api", "weather_and_convert", "converter", "date_time", "rag"}

    if tool_name not in allowed_tools:
        tool_name = "rag"
        reason = f"Fallback to rag because DSPy returned an unknown tool:{tool_name}"
    return {
        "tool_name": tool_name,
        "reason": reason
    }

In [ ]:
def execute_tool_dspy(query):
    decision = decide_tool_dspy(query)
    tool_name = decision["tool_name"]
    reason = decision["reason"]

    thought = f"I analyzed the question and decided that the best tool is: {tool_name}. Reason: {reason}"

    if tool_name == "weather_api":
        city = extract_city(query)
        lat, lon = get_coordinates(city)
        action = f"Calling weather API tool for city: {city}"

        if lat is not None and lon is not None:
            observation = {
                "city": city,
                "weather_data": get_weather_summary(lat, lon)
            }
        else:
            observation = {"error": f"Could not find coordinates for city: {city}"}

    elif tool_name == "weather_and_convert":
        city = extract_city(query)
        lat, lon = get_coordinates(city)
        action = f"Calling weather API for city: {city}, then converting temperature to Fahrenheit"

        if lat is not None and lon is not None:
            weather = get_weather_summary(lat, lon)
            temp_c = weather.get("temperature")

            if temp_c is not None:
                temp_f = celsius_to_fahrenheit(temp_c)
                observation = {
                    "city": city,
                    "temperature_celsius": temp_c,
                    "temperature_fahrenheit": temp_f,
                    "windspeed": weather.get("windspeed"),
                    "winddirection": weather.get("winddirection"),
                    "weathercode": weather.get("weathercode"),
                    "time": weather.get("time")
                }
            else:
                observation = {"error": f"Temperature not found in weather API response for city: {city}"}
        else:
            observation = {"error": f"Could not find coordinates for city: {city}"}

    elif tool_name == "converter":
        action = "Using converter tool"

        q = query.lower()
        numbers = [float(word) for word in q.replace("?", "").split() if word.replace(".", "", 1).isdigit()]

        if numbers:
            value = numbers[0]

            if "celsius" in q and "fahrenheit" in q:
                observation = {
                    "input_celsius": value,
                    "output_fahrenheit": celsius_to_fahrenheit(value)
                }
            elif "fahrenheit" in q and "celsius" in q:
                observation = {
                    "input_fahrenheit": value,
                    "output_celsius": fahrenheit_to_celsius(value)
                }
            else:
                observation = {"error": "Could not determine conversion direction."}
        else:
            observation = {"error": "No numeric value found in the query."}

    elif tool_name == "date_time":
        action = "Using date/time tool"
        observation = get_current_datetime()

    else:
        action = "Using RAG retriever tool"
        observation = rag_retrieve(query)

    return {
        "tool_name": tool_name,
        "thought": thought,
        "action": action,
        "observation": observation
    }

In [ ]:
def run_agent_dspy(query):
    tool_result = execute_tool_dspy(query)

    result = weather_responder(
        query=query,
        thought=tool_result["thought"],
        action=tool_result["action"],
        observation=str(tool_result["observation"])
    )

    final_answer = str(result.final_answer).strip()

    return {
        "query": query,
        "tool_name": tool_result["tool_name"],
        "thought": tool_result["thought"],
        "action": tool_result["action"],
        "observation": tool_result["observation"],
        "final_answer": final_answer
    }

In [ ]:
def print_agent_log_dspy(agent_result):
    print("=" * 80)
    print("USER QUESTION:")
    print(agent_result["query"])
    print("-" * 80)
    print("TOOL USED:")
    print(agent_result["tool_name"])
    print("-" * 80)
    print("THOUGHT:")
    print(agent_result["thought"])
    print("-" * 80)
    print("ACTION:")
    print(agent_result["action"])
    print("-" * 80)
    print("OBSERVATION:")
    print(agent_result["observation"])
    print("-" * 80)
    print("FINAL ANSWER:")
    print(agent_result["final_answer"])
    print("=" * 80)

In [ ]:
test_questions = [
    "What is historical weather data?",
    "What is the weather in Riyadh now?",
    "Convert 20 celsius to fahrenheit",
    "What time is it now?",
    "What is the weather in Paris now and convert it to fahrenheit?"
]

for q in test_questions:
    print("Question:", q)
    print(decide_tool_dspy(q))
    print("-" * 80)

Question: What is historical weather data?
{'tool_name': 'rag', 'reason': 'The question about historical weather data is more suited to a research and general knowledge inquiry rather than needing specific weather data retrieval or conversion, so the rag tool, which can provide information and context, is the best choice.'}
--------------------------------------------------------------------------------
Question: What is the weather in Riyadh now?
{'tool_name': 'weather_api', 'reason': 'The weather_api is the best choice for providing real-time weather information, such as current conditions in Riyadh.'}
--------------------------------------------------------------------------------
Question: Convert 20 celsius to fahrenheit
{'tool_name': 'converter', 'reason': 'The converter tool is the best choice because the query is specifically asking for a temperature conversion from Celsius to Fahrenheit, which falls directly under its functionality.'}
------------------------------------------

In [ ]:
result1 = run_agent_dspy("What is historical weather data?")
print_agent_log_dspy(result1)

USER QUESTION:
What is historical weather data?
--------------------------------------------------------------------------------
TOOL USED:
rag
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: rag. Reason: The question about historical weather data is more suited to a research and general knowledge inquiry rather than needing specific weather data retrieval or conversion, so the rag tool, which can provide information and context, is the best choice.
--------------------------------------------------------------------------------
ACTION:
Using RAG retriever tool
--------------------------------------------------------------------------------
OBSERVATION:
[{'content': 'ces The Historical Weather API is based on reanalysis datasets and uses a combination of weather station, aircraft, buoy, radar, and satellite observations to create a comprehensive record of past weather conditions. These 

In [ ]:
result2 = run_agent_dspy("What is the weather in Riyadh now?")
print_agent_log_dspy(result2)

USER QUESTION:
What is the weather in Riyadh now?
--------------------------------------------------------------------------------
TOOL USED:
weather_api
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: weather_api. Reason: The weather_api is the best choice for providing real-time weather information, such as current conditions in Riyadh.
--------------------------------------------------------------------------------
ACTION:
Calling weather API tool for city: Riyadh
--------------------------------------------------------------------------------
OBSERVATION:
{'city': 'Riyadh', 'weather_data': {'temperature': 18.0, 'windspeed': 3.6, 'winddirection': 37, 'weathercode': 1, 'time': '2026-04-07T01:30'}}
--------------------------------------------------------------------------------
FINAL ANSWER:
The current weather in Riyadh is 18.0°C with a windspeed of 3.6 km/h coming from the northeast.

In [ ]:
result3 = run_agent_dspy("Convert 20 celsius to fahrenheit")
print_agent_log_dspy(result3)

USER QUESTION:
Convert 20 celsius to fahrenheit
--------------------------------------------------------------------------------
TOOL USED:
converter
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: converter. Reason: The converter tool is the best choice because the query is specifically asking for a temperature conversion from Celsius to Fahrenheit, which falls directly under its functionality.
--------------------------------------------------------------------------------
ACTION:
Using converter tool
--------------------------------------------------------------------------------
OBSERVATION:
{'input_celsius': 20.0, 'output_fahrenheit': 68.0}
--------------------------------------------------------------------------------
FINAL ANSWER:
20 degrees Celsius is equal to 68 degrees Fahrenheit.


In [ ]:
result4 = run_agent_dspy("What time is it now?")
print_agent_log_dspy(result4)

USER QUESTION:
What time is it now?
--------------------------------------------------------------------------------
TOOL USED:
date_time
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: date_time. Reason: The query is asking for the current time, which is best handled by the date_time tool.
--------------------------------------------------------------------------------
ACTION:
Using date/time tool
--------------------------------------------------------------------------------
OBSERVATION:
{'date': '2026-04-07', 'time': '04:37:52', 'timezone': 'Asia/Riyadh'}
--------------------------------------------------------------------------------
FINAL ANSWER:
The current time is 04:37:52 in the Asia/Riyadh timezone.


In [ ]:
result5 = run_agent_dspy("What is the weather in Paris now and convert it to fahrenheit?")
print_agent_log_dspy(result5)

USER QUESTION:
What is the weather in Paris now and convert it to fahrenheit?
--------------------------------------------------------------------------------
TOOL USED:
weather_and_convert
--------------------------------------------------------------------------------
THOUGHT:
I analyzed the question and decided that the best tool is: weather_and_convert. Reason: This tool is ideal because it provides current weather information and can also convert temperatures into Fahrenheit, which is required by the query.
--------------------------------------------------------------------------------
ACTION:
Calling weather API for city: Paris, then converting temperature to Fahrenheit
--------------------------------------------------------------------------------
OBSERVATION:
{'city': 'Paris', 'temperature_celsius': 10.5, 'temperature_fahrenheit': 50.9, 'windspeed': 3.7, 'winddirection': 101, 'weathercode': 0, 'time': '2026-04-07T01:30'}
-------------------------------------------------------